# The generative model murder mystery

**IDETC-CIE 2026 · EngiBench hands-on workshop**

*Ten generative models were trained on the same topology optimization problem. The best-performing model is hidden among the suspects. Your task is to identify it from the evidence.*

**Before you edit anything:** File → Save a copy in Drive. Opens read-only from GitHub.

---
## Setup

**About two minutes.** Run the next two cells, then carry on.

In [ ]:
import sys
from pathlib import Path

if "google.colab" in sys.modules:
    setup_marker = Path("/content/.idetc26_setup_v1")
    if setup_marker.exists():
        print("Setup already complete.")
    else:
        %pip install -q "git+https://github.com/IDEALLab/EngiOpt.git@feat/idetc26-workshop"
        # PyPI's engibench is 0.2.0 and so is the build from main, so pip calls
        # the requirement satisfied and leaves photonics2d at v0. Force the
        # GitHub version into the current environment before EngiBench is first
        # imported by the next cell.
        %pip install -q --force-reinstall --no-deps "engibench[all] @ git+https://github.com/IDEALLab/EngiBench.git@main"

        import sysconfig

        installed = Path(sysconfig.get_paths()["purelib"]) / "engibench/problems/photonics2d/v1.py"
        if not installed.exists():
            raise RuntimeError("EngiBench installation failed: photonics2d v1 is missing.")

        setup_marker.write_text("EngiOpt feat/idetc26-workshop; EngiBench main\n")
        print("Setup complete. Continue with the next cell.")

In [ ]:
from engiopt.workshops.idetc26 import Case

case = Case.open("beams2d")     # <- the problem you are working on

The cheat sheet. `case.help(full=True)` adds the controls, the space map and every option.

In [ ]:
case.help()

---
# 1 · The scene of the crime (the dataset)

`"train"` — what the models were fitted on. `"test"` — the held-out set they are
scored against. Sliders move the conditions.

In [ ]:
case.show("train")

In [ ]:
case.problem.render(case.designs("test")[0])

---
# 2 · The suspects

The ten models in our lineup -- which one is best?

In [ ]:
case.models()

Use `case.explain()` to open a model's case file.

In [ ]:
case.explain("knn_retrieval")

### The evidence (visualizing our suspects)

Show outputs generated by one model

In [ ]:
case.show("diffusion", n=12)

Two suspects, same brief. Either side can be a model, `"test"`, or `"train"`.

In [ ]:
case.show("cgan_cnn_2d", "test")

`how=` swaps the picture:

| `how=` | draws |
|---|---|
| `"designs"` *(default)* | `n` designs from the suspect you name |
| `"compare"` | the suspects you name on the same briefs, real optimum on top |
| `"conditions"` | each design captioned `asked 0.30 / got 0.41` |
| `"nearest_training"` | each design above its closest training design |
| `"space_map"` | where a suspect's designs sit in a fitted space |

In [ ]:
case.show("knn_retrieval", "vqgan", how="compare", n=3)

---
# 3 · The interrogation/questions

Each metric examines a different aspect of the model: cost, similarity, novelty, diversity, feasibility, or performance.

In [ ]:
case.metrics()

## Cost — how expensive is the model?

If training or running the model costs nearly as much as the traditional optimizer, does the model offer a practical advantage? Is it worth it?

Cost metrics: `train_minutes`, `gen_seconds`, `params`

In [ ]:
case.evaluate("train_minutes").round(3)

## Similarity — does it look like the real thing?

How close do generated designs resemble traditionally optimized designs?

Similarity metrics: `mmd`, `pixel_paired_distance`

In [ ]:
case.evaluate(["mmd", "pixel_paired_distance"]).round(4)

## Novelty — Are the designs new or memorized?

Values around 1 are comparable to held-out designs, while values near 0 suggest
memorization. Values far above 1 may indicate out-of-distribution outputs;
random noise can also appear highly novel.

In [ ]:
case.evaluate("novelty_ratio").round(3)

By eye — each design above the closest thing to it in the training set.

In [ ]:
case.show("knn_retrieval", how="nearest_training")

## Diversity — How different are our generated designs from each other?

Diversity scores need reference points. Use `controls=True` to include three known controls.


| control | what it means |
|---|---|
| `collapsed` | an averaged design repeated 50 times |
| `noise_doped` | real optima plus Gaussian noise |
| `volume_only` | random design satisfying only the volume constraint|

Diversity metrics: `vendi`, `dpp`

In [ ]:
case.evaluate("pixel_vendi", controls=True).round(3)

In [ ]:
case.show("noise_doped")   # controls can be looked at like any other model

## Obedience — did it meet the constraints?

The design is not valid if it doesn't meet its budget

`cond_err` — mean error in the specified conditions (e.g., volume fraction)
`viol` — fraction of designs exceeding the permitted tolerance

In [ ]:
case.evaluate(["cond_err", "viol"], controls=True).round(4)

In [ ]:
case.show("gan_cnn_2d", how="conditions")

## Performance — is the design actually any good?

Three optimality gaps, measured by re-running the optimizer from each generated
design. Lower is better for all three.

| | column | asks |
|---|---|---|
| **IOG** — initial | `iog` | how far the design starts from the reference optimum |
| **COG** — cumulative | `cog` | how much work the optimizer does getting there |
| **FOG** — final | `fog` | where it ends up |

Each has a median twin — `iog_median`, `cog_median`, `fog_median` — over the same
50 designs. A single design the optimizer cannot rescue has an effectively
unbounded gap, so a mean of 50 is set by its worst member: on `beams2d` three
models report mean IOG of 818, 50 and 1.5e8 while all three finish at the same
FOG. Rank on the mean and you have ranked one design.

These are the columns that call the simulator, so they are the expensive ones.

In [ ]:
case.evaluate("cog", models=["knn_retrieval", "cgan_cnn_2d"]).round(3)

---
# 4 · The same questions, somewhere other than pixels

Every similarity/distance used so far compared designs in pixel space. We can also project to featural spaces. Here we compute the corresponding metrics in a PCA subspace or through a learned autoencoder latent space

| the question | pixels | PCA subspace | learned latent |
|---|---|---|---|
| does it look real? | `mmd` | `pca_mmd` | `lv_mmd` |
| how close to the right answer? | `pixel_paired_distance` | `pca_paired_distance` | `lv_paired_distance` |
| is it copying? | `novelty_ratio` | `pca_novelty` | `lv_novelty` |
| how many distinct designs? | `pixel_vendi` | `pca_vendi` | `lv_vendi` |
| did it cover the data? | — | `pca_coverage` | `lv_coverage` |

In [ ]:
case.show(case.evaluate(["mmd", "pca_mmd", "lv_mmd"]))

For this comparison, PCA is matched to the active dimensionality of the
least-volume autoencoder. The `lv_...` metrics use the encoded latent space of
`constrained_plvae_2d`.

In [ ]:
case.latent_space()     # which autoencoder, how wide, and the PCA width matched to it

### Looking at the space

In [ ]:
case.show("knn_retrieval", "test", how="space_map")

Same designs projected in linear PCA space. Nothing about the model changed; differences between plots come from the representation.

In [ ]:
case.show("knn_retrieval", "test", how="space_map", space="pca")

---
# 5 · Visualizing a board

In [ ]:
board = case.evaluate(["mmd", "pixel_vendi", "iog_median"])
case.show(board)

---
# 6 · Your accusation

Choose the model you would use and be ready to explain your choice to the room:

1. **Who did it.** The model you think is the best.
2. **On what evidence.** Why do you think so?
3. **What you could not rule out.** What additional evidence would make your decision easier?

The cell below is yours to build with.

Remember `case.help()`